# Part B - Custom Retrieval System
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  

This notebook handles the implementation of the retrieval system, including embeddings, vector storage, and hybrid search.

In [7]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# 1. Load Chunks
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks
texts = [c['text'] for c in all_chunks]

# 2. Generate Embeddings (Task 1)
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, show_progress_bar=True)

# 3. FAISS Vector Store (Task 2)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
os.makedirs('indexes', exist_ok=True)
faiss.write_index(index, 'indexes/rag_index.faiss')

# 4. BM25 Setup
tokenized_corpus = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Retrieved {len(all_chunks)} chunks.")
print(f"FAISS index saved with {index.ntotal} vectors.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1553.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 31/31 [02:33<00:00,  4.94s/it]


Retrieved 992 chunks.
FAISS index saved with 992 vectors.


### Hybrid Search Implementation (Task 3 & 4)

In [8]:
def hybrid_search(query, k=3, alpha=0.5):
    # Vector Search
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    
    # BM25 Search
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i:
                v_score = 1 / (1 + distances[0][rank])
                break
        
        score = (alpha * v_score) + ((1 - alpha) * (bm25_scores[i] / max_bm25))
        if score > 0:
            combined.append({'chunk': chunk, 'score': score})
            
    return sorted(combined, key=lambda x: x['score'], reverse=True)[:k]

# Verification
test_query = "How many votes did Nana Addo get in the Ahafo region?"
results = hybrid_search(test_query)
for r in results:
    print(f"Score: {r['score']:.4f} | {r['chunk']['text']}\n")

Score: 0.8650 | In the 2008 Ghana election, Nana Addo from NPP got 392,588 votes (50.56%) in Brong Ahafo Region.

Score: 0.8564 | In the 2012 Ghana election, Nana Akufo Addo from NPP got 115,226 votes (50.78%) in Ahafo Region.

Score: 0.8556 | In the 2016 Ghana election, Nana Akufo Addo from NPP got 123,139 votes (55.11%) in Ahafo Region.



### Critical Task: Failure Case & Fix

**Problem Statement:** Pure vector (semantic) search can fail when a query contains specific acronyms that are semantically similar to other acronyms. The embedding model for `all-MiniLM-L6-v2` encodes political party abbreviations like `NDC` and `NDP` into very similar vector spaces because they are short, rare tokens. This causes the model to **retrieve results for the wrong party**.

**Failure Query:** `"What were the NDC votes in Savannah in 2020?"`

- **Expected:** John Dramani Mahama (NDC) result for the Savannah Region
- **Failure:** Vector search returns an NDP (Nana Konadu) result instead — a completely different party!

In [9]:
def pure_vector_search(query, k=3):
    """Pure semantic vector search only — no keyword component."""
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k)
    return [(all_chunks[i], float(distances[0][j])) for j, i in enumerate(indices[0])]

# This is the critical failure query
failure_query = "What were the NDC votes in Savannah in 2020?"

print("--- FAILURE CASE: Pure Vector Search ---")
print(f'Query: "{failure_query}"\n')
v_results = pure_vector_search(failure_query)
for i, (res, dist) in enumerate(v_results):
    print(f"{i+1}. [L2 dist={dist:.4f}] {res['text']}")
print("\n>>> Vector Search FAILED: It retrieved NDP party results instead of NDC!")
print("    Root Cause: 'NDC' and 'NDP' have nearly identical embedding vectors")
print("    (short acronyms are poorly differentiated by sentence-transformer models)\n")

print("--- FIX: Hybrid Search (BM25 + Vector) ---")
print(f'Query: "{failure_query}"\n')
h_results = hybrid_search(failure_query, k=3)
for i, res in enumerate(h_results):
    print(f"{i+1}. [score={res['score']:.4f}] {res['chunk']['text']}")
print("\n>>> Hybrid Search FIXED: BM25 matched 'NDC' keyword exactly, overriding the semantic confusion.")

--- FAILURE CASE: Pure Vector Search ---
Query: "What were the NDC votes in Savannah in 2020?"

1. [L2 dist=0.7447] In the 2020 Ghana election, Nana Konadu Agyeman Rawlings from NDP got 279 votes (0.12%) in Savannah Region.
2. [L2 dist=0.8188] In the 2020 Ghana election, John Dramani Mahama from NDC got 144,244 votes (62.97%) in Savannah Region.
3. [L2 dist=0.8712] In the 2020 Ghana election, Nana Konadu Agyeman Rawlings from NDP got 117 votes (0.03%) in Western North Region.

>>> Vector Search FAILED: It retrieved NDP party results instead of NDC!
    Root Cause: 'NDC' and 'NDP' have nearly identical embedding vectors
    (short acronyms are poorly differentiated by sentence-transformer models)

--- FIX: Hybrid Search (BM25 + Vector) ---
Query: "What were the NDC votes in Savannah in 2020?"

1. [score=0.7749] In the 2020 Ghana election, John Dramani Mahama from NDC got 144,244 votes (62.97%) in Savannah Region.
2. [score=0.6952] In the 2020 Ghana election, Nana Konadu Agyeman Rawlings

**Root Cause Analysis:**

| | Pure Vector Search | Hybrid Search (Fix) |
|---|---|---|
| Method | Semantic similarity only | BM25 keywords + Vector semantics |
| NDC vs NDP | Treated as nearly identical (similar embeddings) | BM25 distinguishes them as exact string matches |
| Top Result | ❌ NDP (Nana Konadu) — Wrong party | ✅ NDC (John Dramani Mahama) — Correct |

**Conclusion:** The hybrid search system resolves this failure by using BM25's exact keyword matching to **anchor** the search to the specific acronym `NDC`, while the vector component still provides semantic understanding of words like `votes`, `Savannah`, and `2020`. This is exactly why a hybrid approach is superior to either method alone.